In [ ]:
import pandas as pd
import psycopg
from psycopg.rows import dict_row
import tomllib
from rds_chat_analysis import NOTEBOOK_DIR
from psycopg.pq.misc import connection_summary

In [ ]:
from rds_chat_analysis.vector_db import setup_db, setup_indices
from rds_chat_analysis.vector_store_utils import (
    add_embeddings_to_db,
    build_vector_store_query,
)

## Setup

Before running this notebook, make sure your mock and private postgres databases are up,
and the config_mock.toml and config_private.toml are configured correctly.

For a local setup, you can use:
```bash
# Launch mock and private postgres locally
docker compose up -d
```

## Configuration

In [ ]:
MOCK_EMBEDDINGS_DIR = NOTEBOOK_DIR / "v2" / "embeddings_1000" / "mock"
PRIVATE_EMBEDDINGS_DIR = NOTEBOOK_DIR / "v2" / "embeddings_1000" / "private"

MOCK_CONFIG_PATH = NOTEBOOK_DIR / "v2" / "config_mock.toml"
PRIVATE_CONFIG_PATH = NOTEBOOK_DIR / "v2" / "config_private.toml"

In [ ]:
with open(MOCK_CONFIG_PATH, "rb") as f:
    mock_config = tomllib.load(f)
with open(PRIVATE_CONFIG_PATH, "rb") as f:
    private_config = tomllib.load(f)

## Connect to DB
These are the directories that contain your embeddings. See the `01-prepare-data.ipynb` notebook for the expected structure and schema.

In [ ]:
mock_db_config = mock_config["db"]
private_db_config = private_config["db"]

mock_db_settings = {
    "host": mock_db_config["postgres_host"],
    "port": mock_db_config["postgres_port"],
    "dbname": mock_db_config["postgres_db"],
    "user": mock_db_config["postgres_user"],
    "password": mock_db_config["postgres_password"],
    "sslmode": mock_db_config["postgres_sslmode"],
}

private_db_settings = {
    "host": private_db_config["postgres_host"],
    "port": private_db_config["postgres_port"],
    "dbname": private_db_config["postgres_db"],
    "user": private_db_config["postgres_user"],
    "password": private_db_config["postgres_password"],
    "sslmode": private_db_config["postgres_sslmode"],
}

mock_keepalive = mock_db_config["keepalive_settings"]
private_keepalive = private_db_config["keepalive_settings"]

mock_conn = psycopg.connect(
    **mock_db_settings,
    row_factory=dict_row,
    **mock_keepalive,
)
private_conn = psycopg.connect(
    **private_db_settings,
    row_factory=dict_row,
    **private_keepalive,
)

print(f"Connected to Mock DB: {connection_summary(mock_conn.pgconn)}")
print(f"Connected to Private DB: {connection_summary(private_conn.pgconn)}")

## Setup DB schema

In [ ]:
# Set vector index settings
# NOTE: before using, be sure to allowlist `pg_diskann` and `vector` extensions in Azure
# DiskANN is only available in Azure flexible postgres, supports >> 5 million rows
# VECTOR_INDEX_SETTINGS = {
#     "index_type": "diskann",
#     "index_params": {
#         "max_neighbors": 32,
#         "l_value_ib": 100,
#     },
#     "distance_opclass": "vector_cosine_ops",
# }

# Alternative: HNSW for any other postgres with pgvector enabled, supports <= 10 million rows and has some minor inconsistencies with metadata post-filtering
VECTOR_INDEX_SETTINGS = {
    "index_type": "hnsw",
    "index_params": {"m": 16, "ef_construction": 64},
    "distance_opclass": "vector_cosine_ops",
}

In [ ]:
# Setup DB and schema
# - enable correct extensions (pgvector, pg_diskann)
# - create table for embeddings

setup_db(
    conn=mock_conn,
    table_name="log_embeddings",
    embedding_size=768,  # IMPORTANT: should match the embedding_size of your embedder
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    overwrite_existing=True,  # IMPORTANT: this will drop the table if it exists, and remove all your data.
)

setup_db(
    conn=private_conn,
    table_name="log_embeddings",
    embedding_size=768,  # IMPORTANT: should match the embedding_size of your embedder
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    overwrite_existing=True,  # IMPORTANT: this will drop the table if it exists, and remove all your data.
)

# Add data to DB

In [ ]:
# Add mock embeddings to mock DB
add_embeddings_to_db(
    conn=mock_conn,
    table_name="log_embeddings",
    embeddings_dir=MOCK_EMBEDDINGS_DIR,
    batch_size=64,
)

# Setup indices on mock DB (vector index + metadata index)
setup_indices(
    conn=mock_conn,
    table_name="log_embeddings",
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    overwrite_existing=True,  # IMPORTANT: this will drop the index if it exists, and build a new one.
    pg_maintenance_mem="2GB",  # 4 workers * 2GB = 8GB RAM for building the vector index.
    pg_parallel_workers=4,
)

In [ ]:
# Add private embeddings to private DB
# NOTE: for large datasets, inserting and building the vector index can take a long time. Please keep the notebook open until this cell is done.
# ~1-2 hours to insert and build build for 2.5 million rows with 4 workers, 2GB RAM each
add_embeddings_to_db(
    conn=private_conn,
    table_name="log_embeddings",
    embeddings_dir=PRIVATE_EMBEDDINGS_DIR,
    batch_size=64,
)

setup_indices(
    conn=private_conn,
    table_name="log_embeddings",
    vector_index_settings=VECTOR_INDEX_SETTINGS,
    overwrite_existing=True,  # IMPORTANT: this will drop the index if it exists, and build a new one.
    pg_maintenance_mem="2GB",  # 4 workers * 2GB = 8GB RAM for building the vector index
    pg_parallel_workers=4,
)

In [ ]:
with private_conn.cursor() as cur_private:
    cur_private.execute("SELECT COUNT(*) FROM log_embeddings")
    count_private = cur_private.fetchone()["count"]
    print(f"Private db size: {count_private} messages")

with mock_conn.cursor() as cur_mock:
    cur_mock.execute("SELECT COUNT(*) FROM log_embeddings")
    count_mock = cur_mock.fetchone()["count"]
    print(f"Mock db size: {count_mock} messages")

# Test if everything works

In [ ]:
from rds_chat_analysis.utils import load_embedder_from_config

embedder = load_embedder_from_config(mock_config)

In [ ]:
# Create an example query
query = "Messages about AI and privacy"
query_embedding = embedder.embed_query(query)

db_query, db_query_params = build_vector_store_query(
    query_embedding=query_embedding,
    table_name="log_embeddings",
    k=5,
    distance_threshold=0.5,
    filters={"language": "English"},
)

with private_conn.cursor() as cur:
    cur.execute(db_query, db_query_params)
    results = cur.fetchall()

pd.json_normalize(results)  # json_normalize flattens metadata to individual columns